# EchoDepthNet (Residual FG-EDFM) — One-Click Cloud GPU Training on NYU Depth V2

Run all cells (`Runtime -> Run all`) on a free Google Colab T4 GPU / Kaggle GPU / AutoDL GPU to:
1. Clone `EchoSight` (`https://github.com/zhud41343-alt/echosight`)
2. Train **Full EchoDepthNet (+Residual FG-EDFM)** and **Ablation Baseline (w/o FG-EDFM)** on **NYU Depth V2** with identical deterministic seed (`seed=42`)
3. Generate the publication-ready **Ablation Table (`ablation_report.md`)**, **4-Panel Figure (`paper_figure_preview.png`)**, and **ONNX Edge Model (`echodepth_net.onnx`)**

In [ ]:
!nvidia-smi
!git clone https://github.com/zhud41343-alt/echosight.git || (cd echosight && git pull)
%cd echosight/research
!pip install -q onnx onnxruntime matplotlib tqdm pillow h5py datasets

In [ ]:
# Train Full EchoDepthNet (+Residual FG-EDFM) on NYU Depth V2 (ImageNet pretrained MobileNetV3-Small)
!python train.py --nyu-hf --pretrained --epochs 30 --batch-size 32 --img-size 256 --seed 42 --out-dir outputs

In [ ]:
# Train Ablation Baseline (w/o FG-EDFM) under identical seed and hyperparameters
!python train.py --nyu-hf --pretrained --epochs 30 --batch-size 32 --img-size 256 --seed 42 --ablation-no-edge --out-dir outputs

In [ ]:
# Evaluate, generate 4-panel figure, and export ONNX edge deployment model
!python evaluate_ablation.py
!python export_onnx.py --checkpoint outputs/echodepth_best.pt --output outputs/echodepth_net.onnx
from IPython.display import Image, display, Markdown
display(Markdown(open('outputs/ablation_report.md', encoding='utf-8').read()))
display(Image('outputs/paper_figure_preview.png'))

In [ ]:
# Package outputs for download (Colab)
!zip -r echodepth_nyu_results.zip outputs/ablation_report.md outputs/paper_figure_preview.png outputs/echodepth_net.onnx
try:
    from google.colab import files
    files.download('echodepth_nyu_results.zip')
except ImportError:
    print('Saved echodepth_nyu_results.zip in current directory.')